# Kaggle Train With Fast Git Sync

Notebook này dùng GPU Kaggle để train Tiny YOLO from scratch. Nếu repo đã có trong `/kaggle/working/XLA` thì chỉ đồng bộ code mới từ GitHub; nếu chưa có thì clone lần đầu.

In [ ]:
REPO_URL = "https://github.com/huyvanzzz/XLA.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working/XLA"

# Sửa đường dẫn này theo Kaggle Dataset của bạn.
KAGGLE_PUBLIC_DIR = "/kaggle/input/xla-object-detection/public"

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
work_path = Path(WORK_DIR)

if (work_path / ".git").exists():
    print("Repo exists, syncing latest code...")
    subprocess.run(["git", "-C", WORK_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    if work_path.exists():
        shutil.rmtree(work_path)
    print("Repo not found, cloning...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)

os.chdir(WORK_DIR)
print("Using repo at", WORK_DIR)
!git log --oneline -1

In [ ]:
# Kaggle có thể dùng PyTorch quá mới không hỗ trợ Tesla P100 (sm_60).
# Bản này tương thích P100 và vẫn chạy tốt trên GPU Kaggle phổ biến.
!python -m pip uninstall -y -q torch torchvision torchaudio
!python -m pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch==2.4.1+cu121
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import shutil

src_public = Path(KAGGLE_PUBLIC_DIR)
dst_public = Path(WORK_DIR) / "public"

if src_public.exists():
    if dst_public.exists():
        shutil.rmtree(dst_public)
    shutil.copytree(src_public, dst_public)
    print("Copied dataset from", src_public)
elif dst_public.exists():
    print("Using existing public/ inside repo")
else:
    raise FileNotFoundError(f"Cannot find dataset at {src_public}. Upload public/ as a Kaggle Dataset and update KAGGLE_PUBLIC_DIR.")

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/ \
  --config ./configs/default.yaml

In [ ]:
!python predict.py \
  --image_dir ./public/val/images \
  --output ./val_predictions.json \
  --checkpoint ./models/best.pth \
  --config ./configs/default.yaml \
  --batch_size 16

!python public/tools/evaluate_predictions.py \
  --ground_truth ./public/annotations/val.json \
  --predictions ./val_predictions.json \
  --output ./val_score.json

!cat ./val_score.json

In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path("/kaggle/working/artifacts")
artifact_dir.mkdir(exist_ok=True)
for name in ["best.pth", "last.pth"]:
    src = Path("./models") / name
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
for name in ["val_predictions.json", "val_score.json"]:
    src = Path(name)
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
print("Artifacts:", sorted(p.name for p in artifact_dir.iterdir()))